In [ ]:
! pip install -q -U google-genai anthropic together

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 956.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.8/728.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.8/330.8 kB 6.0 MB/s eta 0:00:00


In [ ]:
models_to_evaluate = [
    "gemini-3-pro-preview",
    "gemini-3-flash-preview",
    "gemini-2.5-flash",

    "claude-haiku-4-5-20251001",
    "claude-opus-4-6",

    "deepseek-ai/DeepSeek-R1",
    "openai/gpt-oss-120b",
    "OpenAI/gpt-oss-20B",
    "zai-org/GLM-4.7",
    "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
    "Qwen/Qwen3-Next-80B-A3B-Thinking",

    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",

    "gpt-4.1-2025-04-14",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4.1-nano-2025-04-14",

    "gpt-5.2-2025-12-11",

    "gpt-5-2025-08-07",
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
]

# Batch inference functions

In [ ]:
from pydantic import BaseModel, ConfigDict
from openai import OpenAI
from google.colab import userdata
from google.genai import types
from google import genai
from anthropic import Anthropic, transform_schema
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from together import Together
import json
import os
import time
import uuid
import io

os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_AI_STUDIO_KEY')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('CLAUDE_PERSONAL_KEY')

gemini_client = genai.Client()
anthropic_client = Anthropic()
together_client = Together(api_key=userdata.get('TOGETHER_KEY'))
openai_client = OpenAI(api_key=userdata.get('ELM_API_KEY'))

class MCQResponse(BaseModel):
    model_config = ConfigDict(
          title="Multiple choice question response",
          json_schema_extra={
              "description": "The response to the multiple choice question given as the single letter which corresponds to the correct answer."
          }
    )
    single_letter_answer: str

class TranslationResponse(BaseModel):
    model_config = ConfigDict(
          title="Translation response",
          json_schema_extra={
              "description": "Response with the requested translation of the given texts."
          }
    )
    translation_text: str

### Send batch functions

In [ ]:
def make_batch_jsonl_inputs(sys_msgs, prompts, model, output_schema):
    schema_json = output_schema.model_json_schema()
    schema_json["additionalProperties"] = False
    return [{
        "custom_id": f"request-{i}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "messages": [
              {
                "role": "system",
                "content": sys_msg,
              },
              {
                "role": "user",
                "content": prompt,
              }
            ],
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                  "name": "output_format",
                  "schema": schema_json,
                  "strict": True,
                }
            }
        }
    } for i, (sys_msg, prompt) in enumerate(zip(sys_msgs, prompts))]

def make_gemini_batch_jsonl_inputs(sys_msgs, prompts, model, output_schema):
    output_schema_json = output_schema.model_json_schema()
    del output_schema_json["title"]
    del output_schema_json["description"]
    output_schema_json["type"] = output_schema_json["type"].upper()
    output_schema_json["properties"] = {
        k: {"type": v["type"].upper()} for k, v in output_schema_json["properties"].items()
    }
    return [{
        "key": f"request-{i}",
        "request": {
            "contents": [{"parts": [{"text": prompt}]}],
            "system_instruction": {"parts": [{"text": sys_msg}]},
            "generation_config": {
                "response_mime_type": "application/json",
                "response_schema": output_schema_json,
            }
            }
        } for i, (sys_msg, prompt) in enumerate(zip(sys_msgs, prompts))]

def make_batch_buffer(sys_msgs, prompts, model, output_schema):

    batch_inputs = make_batch_jsonl_inputs(
        sys_msgs, prompts, model, output_schema
    )

    buffer = io.BytesIO()

    for line in batch_inputs:
        json_line = json.dumps(line) + "\n"
        buffer.write(json_line.encode("utf-8"))

    buffer.seek(0)

    return buffer

def send_together_batch(sys_msgs, prompts, model, output_schema):

    batch_inputs = make_batch_jsonl_inputs(
        sys_msgs, prompts, model, output_schema
    )

    jsonl_path = f"together_{str(time.time())}.jsonl"

    with open(jsonl_path, "a") as f:
        for line in batch_inputs:
            f.write(json.dumps(line) + "\n")

    file_resp = together_client.files.upload(
        file=jsonl_path, purpose="batch-api", check=False
    )
    file_id = file_resp.id

    batch = together_client.batches.create(
        input_file_id=file_id, endpoint="/v1/chat/completions"
    )

    return batch.job.id

def send_openai_batch(sys_msgs, prompts, model, output_schema):
    batch_buffer = make_batch_buffer(
        sys_msgs, prompts, model, output_schema
    )

    batch_input_file = openai_client.files.create(
        file=("batch_requests.jsonl", batch_buffer),
        purpose="batch"
    )

    batch_input_file_id = batch_input_file.id

    batch_object = openai_client.batches.create(
        input_file_id=batch_input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    return batch_object.id

def send_gemini_batch(sys_msgs, prompts, model, output_schema):

    batch_inputs = make_gemini_batch_jsonl_inputs(
        sys_msgs, prompts, model, output_schema
    )

    jsonl_path = f"gemini_{str(time.time())}.jsonl"

    with open(jsonl_path, "a") as f:
        for line in batch_inputs:
            f.write(json.dumps(line) + "\n")

    uploaded_file = gemini_client.files.upload(
        file=jsonl_path,
        config=types.UploadFileConfig(display_name=jsonl_path, mime_type='jsonl')
    )

    file_batch_job = gemini_client.batches.create(
        model=model,
        src=uploaded_file.name,
        config={
            'display_name': jsonl_path,
        },
    )

    return file_batch_job.name

def send_claude_batch(sys_msgs, prompts, model, output_schema):
    return anthropic_client.messages.batches.create(
        requests=[
            Request(
                custom_id=str(uuid.uuid4()),
                params=MessageCreateParamsNonStreaming(
                    model=model,
                    max_tokens=9999,
                    system=sys_msg,
                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        },
                    ],
                    output_config={
                        "format": {
                            "type": "json_schema",
                            "schema": transform_schema(output_schema),
                        }
                    }
                )
            ) for sys_msg, prompt in zip(sys_msgs, prompts)
        ]
    ).id


def send_batch(sys_msgs, prompts, model, output_schema):

    if "/" in model:
        return send_together_batch(sys_msgs, prompts, model, output_schema)
    if "gpt" in model.lower():
        return send_openai_batch(sys_msgs, prompts, model, output_schema)
    if "claude" in model.lower():
        return send_claude_batch(sys_msgs, prompts, model, output_schema)
    if "gemini" in model.lower():
        return send_gemini_batch(sys_msgs, prompts, model, output_schema)

### Status function

In [ ]:
def get_batch_status(model, batch_id):
    if "/" in model:
        return together_client.batches.retrieve(batch_id).status
    if "gpt" in model.lower():
        return openai_client.batches.retrieve(batch_id).status
    if "claude" in model.lower():
        return anthropic_client.messages.batches.retrieve(batch_id).processing_status
    if "gemini" in model.lower():
        return gemini_client.batches.get(name=batch_id).state.name

### Retrieve batch results

In [ ]:
import json
from tqdm.auto import tqdm

def parse_json(json_str):
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        print("Json decode error:")
        print(json_str)
        return dict()

def get_openai_batch_results(batch_id):
    output_file_id = openai_client.batches.retrieve(batch_id).output_file_id
    file_response = openai_client.files.content(output_file_id)
    output_lines = [
        json.loads(x)["response"]["body"]["choices"][0]["message"]["content"] for x in file_response.text.splitlines() if x.strip()
    ]
    return [parse_json(x) for x in output_lines]

def get_together_batch_results(batch_id):

    batch = together_client.batches.retrieve(batch_id)
    results = []
    with together_client.files.with_streaming_response.content(
        id=batch.output_file_id
    ) as response:
        content = response.read().decode('utf-8')

        for line in content.strip().split('\n'):
            results.append(json.loads(line) if line else None)

    result_indices = [int(x["custom_id"].split("-")[-1]) for x in results]
    ordered_results = [None] * int(max(result_indices) + 1)

    for i, x in zip(result_indices, results):
        ordered_results[i] = x

    return [parse_json(
        x["response"]["body"]["choices"][0]["message"]["content"]
    ) if x else {} for x in ordered_results]

def get_gemini_batch_results(batch_id):

    result_file_name = gemini_client.batches.get(name=batch_id).dest.file_name
    file_content = gemini_client.files.download(file=result_file_name)

    batch_lines = [
        parse_json(x) for x in file_content.decode('utf-8').strip().split("\n")
    ]

    result_indices = [int(x["key"].split("-")[-1]) for x in batch_lines]
    ordered_results = [None] * int(max(result_indices) + 1)

    for i, x in zip(result_indices, batch_lines):
        ordered_results[i] = x

    batch_results = [
        parse_json(
            x["response"]["candidates"][0]["content"]["parts"][0]["text"]
        ) if bool(
            x is not None
        ) and bool(
            "response" in x.keys()
        ) and bool(
            len(x["response"]["candidates"][0]["content"].keys()) > 0
        ) else None for x in tqdm(ordered_results)
    ]

    return batch_results

def get_claude_batch_results(batch_id):
    results = []

    for result in anthropic_client.messages.batches.results(
        batch_id,
    ):
        if "message" in vars(result.result).keys():
            results.append(parse_json(result.result.message.content[0].text))
        else:
            results.append(dict())
    return results

def retrieve_batch(batch_id, model):
    if "/" in model:
        return get_together_batch_results(batch_id)
    if "gpt" in model.lower():
        return get_openai_batch_results(batch_id)
    if "claude" in model.lower():
        return get_claude_batch_results(batch_id)
    if "gemini" in model.lower():
        return get_gemini_batch_results(batch_id)

make_combined_df = lambda df_list: pd.concat(
    [x.T for x in df_list],
    axis=0
).T

In [ ]:
gemini_results = retrieve_batch(
    "batches/cfusyoo19tsdzgb1adgduu5wv2f9e9n7w9o7",
    "gemini"
)

  0%|          | 0/908 [00:00<?, ?it/s]

In [ ]:
gemini_results[0]

{'translation_text': 'Dubhrach\nBha mi ag innse dhuibh an t-seachdain sa chaidh mu Phàdraig Grannd, no Dubhrach. B\' e seann Sheumasach a bh\' ann. \'S e Pàdraig an duine mu dheireadh a chaochail a bha aig Blàr Chùil Lodair.\n\nÀs dèidh Chùil Lodair bha e a\' fuireach ann an Gleann Leathanaidh. Tha sin ann an Glinn Aonghais. Mun cuairt air 1820 bha dithis fhear ann an Gleann Leathanaidh. B\' e ceannaichean a bh\' annta. Choinnich iad ri Pàdraig. Bha Pàdraig còrr is ceud bliadhna a dh\'aois. Dh\'innis Pàdraig dhaibh mu a bheatha.\n\nDh\'fheuch an dithis fhear ri peinnsean rìoghail fhaighinn do Phàdraig. \'S e sin peinnsean bhon Rìgh – Deòrsa IV. Ach b\' e Seumasach a bh\' ann am Pàdraig fhathast! Chaidh an Rìgh a dh\'Alba ann an 1822. Fhuair Pàdraig cuireadh an Rìgh a choinneachadh ann an Dùn Èideann.\n\nChuir am Morair Panmure aodach gu Pàdraig. Bha an t-aodach iomchaidh airson coinneachadh ris an Rìgh. Ach dhiùlt Pàdraig an t-aodach. Chuir e air an t-aodach a bha air aig Blàr Chùil Lo

In [ ]:
# sys_msgs = ["Answer truthfully", "Answer truthfully"]
# prompts = [
#     "What is the capital of France? A. Lyon or B. Paris",
#     "What is the capital of France? A. London or B. Paris"
# ]
# output_schema = MCQResponse

# send_together_batch(sys_msgs, prompts, "deepseek-ai/DeepSeek-R1", output_schema)
# send_openai_batch(sys_msgs, prompts, "gpt-4o", output_schema)
# print(send_gemini_batch(sys_msgs, prompts, "gemini-2.5-flash", output_schema))
# send_claude_batch(sys_msgs, prompts, "claude-haiku-4-5-20251001", output_schema)

# openai_client.batches.retrieve("batch_698a25c2ed3c819084f0dcff40260b94")
# together_client.batches.retrieve("0a9eb3e5-b069-45d4-8b97-217f6c3d53c8")
# gemini_client.batches.get(name="batches/s6z57ll1dt5k9vnm724o8a1z12ep5rckvhas")
# anthropic_client.messages.batches.retrieve("msgbatch_013BaRtx83Tu7JBSzEHWoDQi")

# get_together_batch_results("0a9eb3e5-b069-45d4-8b97-217f6c3d53c8")
# get_claude_batch_results("msgbatch_013BaRtx83Tu7JBSzEHWoDQi")
# get_gemini_batch_results("batches/s6z57ll1dt5k9vnm724o8a1z12ep5rckvhas")
# get_openai_batch_results("batch_698a25c2ed3c819084f0dcff40260b94")

# gemini_client.batches.get(name="batches/91fwiesxfvwy4trmefus04s0y2ceklwflhwk")

# Formatting functions

In [ ]:
mcq_formatting_msg = "Your answer should be a single letter, no other formatting or words."
open_mcq_system_message = """Based on your general knowledge, answer the question given, giving your answer as a single letter corresponding to one of the 4 given options."""
open_mcq_system_message += "\n" + mcq_formatting_msg

def make_open_mcq_input(line):
    question = line["question"]
    choices = {k[-1]:line[k] for k in sorted(line.keys()) if k.startswith("answer_") and line[k] is not None}
    choices = "\n\n".join(f"{k}. {v}" for k, v in choices.items())
    return f"{question}\n\n{choices}"

def send_open_mcq_batch(ds, model, sys_msg=None):
    prompts = [make_open_mcq_input(line) for line in ds]
    sys_msgs = [open_mcq_system_message if sys_msg is None else sys_msg for _ in ds]
    return send_batch(sys_msgs, prompts, model, MCQResponse)

In [ ]:
closed_mcq_system_message = """Based on the text given, answer the question at the end, giving your answer as a single letter corresponding to one of the 4 given options."""
closed_mcq_system_message += "\n" + mcq_formatting_msg

def make_closed_mcq_input(line):
    text = line["gd_title"] + "\n" + line["gd_text"]
    question = line["question"]
    choices = {k[-1]:line[k] for k in sorted(line.keys()) if k.startswith("answer_") and line[k] is not None}
    choices = "\n\n".join(f"{k}. {v}" for k, v in choices.items())
    return f"{text}\n\n{question}\n\n{choices}"

def send_closed_mcq_batch(ds, model):
    prompts = [make_closed_mcq_input(line) for line in ds]
    sys_msgs = [closed_mcq_system_message for _ in ds]
    return send_batch(sys_msgs, prompts, model, MCQResponse)

In [ ]:
translate_system_message = lambda s, t: f"""Given some text in {s}, translate it to {t}.
Your output should only be the translation, no explanation or other formatting."""

format_article = lambda title, text: f"{title}\n{text}"

def make_translation_input(line, source_lang):
    gd_title = line["gd_title"]
    gd_text = line["gd_text"]
    en_title = line["en_title"]
    en_text = line["en_text"]

    gd_article = format_article(gd_title, gd_text)
    en_article = format_article(en_title, en_text)
    if source_lang == "Scottish Gaelic":
        return gd_article
    if source_lang == "English":
        return en_article

def send_translation_batch(ds, source_lang, target_lang, model):
    prompts = [make_translation_input(line, source_lang) for line in ds]
    sys_msgs = [translate_system_message(source_lang, target_lang) for _ in ds]
    return send_batch(sys_msgs, prompts, model, TranslationResponse)

# Load datasets

In [ ]:
from datasets import load_dataset
manual_ds = load_dataset(
    "ptrdvn/gaelic-bench",
    "gd-manual-questions",
    split="test",
)


README.md: 0.00B [00:00, ?B/s]

gd-manual-questions/test-00000-of-00001.(…):   0%|          | 0.00/18.0k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/123 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset

podcasts_ds = load_dataset(
    "ptrdvn/gaelic-bench",
    "podcasts",
    split="test",
).remove_columns(
    ["gd_audio"]
)

pod_open_ds_dict = {
    "en": load_dataset(
        "ptrdvn/gaelic-bench",
        "en-podcast-cultural-questions",
        split="test",
    ),
    "gd": load_dataset(
        "ptrdvn/gaelic-bench",
        "gd-podcast-cultural-questions",
        split="test",
    ),
}

pod_closed_ds_dict = {
    "en": load_dataset(
        "ptrdvn/gaelic-bench",
        "en-podcast-comprehension-questions",
        split="test",
    ),
    "gd": load_dataset(
        "ptrdvn/gaelic-bench",
        "gd-podcast-comprehension-questions",
        split="test",
    ),
}

manual_ds = load_dataset(
    "ptrdvn/gaelic-bench",
    "gd-manual-questions",
    split="test",
)

ds_dict = {
    "open_questions": pod_open_ds_dict,
    "closed_questions": pod_closed_ds_dict,
    "translation": {
        "gd_to_en": podcasts_ds, "en_to_gd": podcasts_ds,
    },
    "manual_questions": manual_ds,
}

README.md: 0.00B [00:00, ?B/s]

podcasts/test-00000-of-00009.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

podcasts/test-00001-of-00009.parquet:   0%|          | 0.00/395M [00:00<?, ?B/s]

podcasts/test-00002-of-00009.parquet:   0%|          | 0.00/487M [00:00<?, ?B/s]

podcasts/test-00003-of-00009.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1087 [00:00<?, ? examples/s]

en-podcast-comprehension-questions/test-(…):   0%|          | 0.00/1.24M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/908 [00:00<?, ? examples/s]

gd-podcast-comprehension-questions/test-(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
ds_dict

{'open_questions': {'en': Dataset({
      features: ['page_id', 'question', 'correct_ans', 'answer_A', 'answer_B', 'answer_C', 'answer_D'],
      num_rows: 1087
  }),
  'gd': Dataset({
      features: ['page_id', 'question', 'correct_ans', 'answer_A', 'answer_B', 'answer_C', 'answer_D'],
      num_rows: 1087
  })},
 'closed_questions': {'en': Dataset({
      features: ['page_id', 'gd_title', 'gd_text', 'question', 'correct_ans', 'answer_A', 'answer_B', 'answer_C', 'answer_D'],
      num_rows: 908
  }),
  'gd': Dataset({
      features: ['page_id', 'gd_title', 'gd_text', 'question', 'correct_ans', 'answer_A', 'answer_B', 'answer_C', 'answer_D'],
      num_rows: 908
  })},
 'translation': {'gd_to_en': Dataset({
      features: ['page_id', 'subject_type', 'en_title', 'gd_title', 'en_text', 'gd_text'],
      num_rows: 908
  }),
  'en_to_gd': Dataset({
      features: ['page_id', 'subject_type', 'en_title', 'gd_title', 'en_text', 'gd_text'],
      num_rows: 908
  })}}

# Translations

In [ ]:
translation_batch_ids = {}

for translation_task_name in ds_dict["translation"].keys():
    translation_batch_ids[translation_task_name] = {}
    for model_name in models_to_evaluate:
        print(f"{translation_task_name} - {model_name}")
        source_language = "Scottish Gaelic" if translation_task_name == "gd_to_en" else "English"
        target_language = "English" if translation_task_name == "gd_to_en" else "Scottish Gaelic"
        translation_batch_ids[translation_task_name][model_name] = send_translation_batch(
            ds_dict["translation"][translation_task_name], source_language, target_language, model_name
        )

translation_batch_ids

gd_to_en - gemini-3-pro-preview
gd_to_en - gemini-3-flash-preview
gd_to_en - gemini-2.5-flash
gd_to_en - claude-haiku-4-5-20251001
gd_to_en - claude-opus-4-6
gd_to_en - deepseek-ai/DeepSeek-R1


Uploading file together_1770819455.849234.jsonl: 100%|██████████| 2.82M/2.82M [00:00<00:00, 5.35MB/s]


gd_to_en - openai/gpt-oss-120b


Uploading file together_1770819459.5001898.jsonl: 100%|██████████| 2.82M/2.82M [00:00<00:00, 7.54MB/s]


gd_to_en - OpenAI/gpt-oss-20B


...

Uploading file together_1770819518.7937667.jsonl: 100%|██████████| 2.40M/2.40M [00:00<00:00, 6.01MB/s]


en_to_gd - Qwen/Qwen3-Next-80B-A3B-Thinking


Uploading file together_1770819520.8727007.jsonl: 100%|██████████| 2.38M/2.38M [00:00<00:00, 6.76MB/s]


en_to_gd - gpt-4o-2024-08-06
en_to_gd - gpt-4o-mini-2024-07-18
en_to_gd - gpt-4.1-2025-04-14
en_to_gd - gpt-4.1-mini-2025-04-14
en_to_gd - gpt-4.1-nano-2025-04-14
en_to_gd - gpt-5.2-2025-12-11
en_to_gd - gpt-5-2025-08-07
en_to_gd - gpt-5-mini-2025-08-07
en_to_gd - gpt-5-nano-2025-08-07


{'gd_to_en': {'gemini-3-pro-preview': 'batches/vxch3c4ry6t4nsqxbmgdlz7uso8kmoa0xzol',
  'gemini-3-flash-preview': 'batches/n56ofwxvb24d32a8qgmibfmcf7kmppzxodhh',
  'gemini-2.5-flash': 'batches/46vzbms2gydiafke06pyqysbha2lb9zaj3e3',
  'claude-haiku-4-5-20251001': 'msgbatch_01XnFRcTtFFQcMi7jV1humaX',
  'claude-opus-4-6': 'msgbatch_01N4mbyZV5Jf53mrWXsX1yE4',
  'deepseek-ai/DeepSeek-R1': 'bded28b6-4c02-4f7e-ab3f-6ddc19686d1c',
  'openai/gpt-oss-120b': '22db1314-2780-4109-82df-55247e3556f6',
  'OpenAI/gpt-oss-20B': '61fe8fa1-810a-443a-9440-c4f994ee8612',
  'moonshotai/Kimi-K2.5': '6a885162-bbc5-4d59-9c68-5bab9c8bed49',
  'zai-org/GLM-4.7': '3b721272-b317-44b3-8983-deea98005647',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '3f925524-071c-4669-b0b4-ee749e38c82f',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': '4fe0d544-bfc4-49f1-9034-d49e49bf2442',
  'gpt-4o-2024-08-06': 'batch_698c8f97a9b88190829a78294501d05b',
  'gpt-4o-mini-2024-07-18': 'batch_698c8f99916081909fcafab8d4558d58',
  'gpt-4.

In [ ]:
translation_batch_ids = {'gd_to_en': {'gemini-3-pro-preview': 'batches/vxch3c4ry6t4nsqxbmgdlz7uso8kmoa0xzol',
  'gemini-3-flash-preview': 'batches/n56ofwxvb24d32a8qgmibfmcf7kmppzxodhh',
  'gemini-2.5-flash': 'batches/46vzbms2gydiafke06pyqysbha2lb9zaj3e3',
  'claude-haiku-4-5-20251001': 'msgbatch_01XnFRcTtFFQcMi7jV1humaX',
  'claude-opus-4-6': 'msgbatch_01N4mbyZV5Jf53mrWXsX1yE4',
  'deepseek-ai/DeepSeek-R1': 'bded28b6-4c02-4f7e-ab3f-6ddc19686d1c',
  'openai/gpt-oss-120b': '22db1314-2780-4109-82df-55247e3556f6',
  'OpenAI/gpt-oss-20B': '61fe8fa1-810a-443a-9440-c4f994ee8612',
  'moonshotai/Kimi-K2.5': '6a885162-bbc5-4d59-9c68-5bab9c8bed49',
  'zai-org/GLM-4.7': '3b721272-b317-44b3-8983-deea98005647',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '3f925524-071c-4669-b0b4-ee749e38c82f',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': '4fe0d544-bfc4-49f1-9034-d49e49bf2442',
  'gpt-4o-2024-08-06': 'batch_698c8f97a9b88190829a78294501d05b',
  'gpt-4o-mini-2024-07-18': 'batch_698c8f99916081909fcafab8d4558d58',
  'gpt-4.1-2025-04-14': 'batch_698c8f9bad008190a16991708c5e31b3',
  'gpt-4.1-mini-2025-04-14': 'batch_698c8f9d8e588190a4ee64e152b28c17',
  'gpt-4.1-nano-2025-04-14': 'batch_698c8f9f195c8190aed06241de19690c',
  'gpt-5.2-2025-12-11': 'batch_698c8fa137f08190880cebacb2f1b206',
  'gpt-5-2025-08-07': 'batch_698c8fa2a8d0819092905fe94d2d1b57',
  'gpt-5-mini-2025-08-07': 'batch_698c8fa40d9c819087b17e686afd84c7',
  'gpt-5-nano-2025-08-07': 'batch_698c8fa5eccc8190b8139ebeec52a113'},
 'en_to_gd': {'gemini-3-pro-preview': 'batches/cfusyoo19tsdzgb1adgduu5wv2f9e9n7w9o7',
  'gemini-3-flash-preview': 'batches/9zhvgc68d8vgnsyy4e5s06k6dbl4zqp2bujm',
  'gemini-2.5-flash': 'batches/6m7ka5aoup8xa3qcbuohpnmuezk4u6urwx2r',
  'claude-haiku-4-5-20251001': 'msgbatch_015rYAYYXQh8KyMGDkoSp4Yc',
  'claude-opus-4-6': 'msgbatch_01UzZN9KDGuEEVYsnuvjRys7',
  'deepseek-ai/DeepSeek-R1': 'f07eb261-7fc3-46ae-99ed-790ac8cf3679',
  'openai/gpt-oss-120b': '917e41d8-8b59-45ef-b792-131f91b74b46',
  'OpenAI/gpt-oss-20B': '91c49b9c-2081-4a3d-837b-bbf42614b616',
  'moonshotai/Kimi-K2.5': '53b1a790-5fd5-4f66-ad40-b5ade8c60910',
  'zai-org/GLM-4.7': '2db5d2b1-8be2-4560-b2a7-b4c78fd2d51a',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '1f379d03-0029-4b23-bc8d-359266c42aa6',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': 'fe27b0b2-f7f0-43ad-9abb-2a230bddd9cd',
  'gpt-4o-2024-08-06': 'batch_698c8fc4771c8190a3aa0b228a4f857b',
  'gpt-4o-mini-2024-07-18': 'batch_698c8fc618d0819092908640789277e0',
  'gpt-4.1-2025-04-14': 'batch_698c8fc789b881908710c07714080a93',
  'gpt-4.1-mini-2025-04-14': 'batch_698c8fc89fb08190beff56bbd5173998',
  'gpt-4.1-nano-2025-04-14': 'batch_698c8fc9cc7c81908b14bceb67f17e14',
  'gpt-5.2-2025-12-11': 'batch_698c8fcae5ac81909ca75f9b6060f35f',
  'gpt-5-2025-08-07': 'batch_698c8fccf19c81908b85e8391424245c',
  'gpt-5-mini-2025-08-07': 'batch_698c8fce89d48190b4b4d19884618f4c',
  'gpt-5-nano-2025-08-07': 'batch_698c8fcfbea08190a7b71f468c068f2c'}}

In [ ]:
for task, batch_ids in translation_batch_ids.items():
    for model_name, batch_id in batch_ids.items():
        print(f"{task} - {model_name}")
        print(get_batch_status(model_name, batch_id))

gd_to_en - gemini-3-pro-preview
JOB_STATE_SUCCEEDED
gd_to_en - gemini-3-flash-preview
JOB_STATE_SUCCEEDED
gd_to_en - gemini-2.5-flash
JOB_STATE_SUCCEEDED
gd_to_en - claude-haiku-4-5-20251001
ended
gd_to_en - claude-opus-4-6
ended
gd_to_en - deepseek-ai/DeepSeek-R1
COMPLETED
gd_to_en - openai/gpt-oss-120b
COMPLETED
gd_to_en - OpenAI/gpt-oss-20B
COMPLETED
gd_to_en - moonshotai/Kimi-K2.5
FAILED
gd_to_en - zai-org/GLM-4.7
COMPLETED
gd_to_en - meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8
COMPLETED
gd_to_en - Qwen/Qwen3-Next-80B-A3B-Thinking
COMPLETED
gd_to_en - gpt-4o-2024-08-06
completed
gd_to_en - gpt-4o-mini-2024-07-18
completed
gd_to_en - gpt-4.1-2025-04-14
completed
gd_to_en - gpt-4.1-mini-2025-04-14
completed
gd_to_en - gpt-4.1-nano-2025-04-14
completed
gd_to_en - gpt-5.2-2025-12-11
completed
gd_to_en - gpt-5-2025-08-07
completed
gd_to_en - gpt-5-mini-2025-08-07
completed
gd_to_en - gpt-5-nano-2025-08-07
completed
en_to_gd - gemini-3-pro-preview
JOB_STATE_SUCCEEDED
en_to_gd - gem

In [ ]:
import pandas as pd

batch_results = {}

for task, batch_ids in translation_batch_ids.items():
    batch_results[task] = {}
    for model_name, batch_id in batch_ids.items():
        if "moonshotai/Kimi-K2.5" in model_name:
            continue
        print(f"{task} - {model_name}")
        batch_result = retrieve_batch(batch_id, model_name)
        batch_result = [x if isinstance(x, dict) else dict() for x in batch_result]
        model_result_df = pd.DataFrame(
            batch_result
        )
        model_result_df.columns = [f"{c} - {model_name}" for c in model_result_df.columns]
        batch_results[task][model_name] = model_result_df

gd_to_en - gemini-3-pro-preview


  0%|          | 0/908 [00:00<?, ?it/s]

gd_to_en - gemini-3-flash-preview


  0%|          | 0/908 [00:00<?, ?it/s]

gd_to_en - gemini-2.5-flash


...

en_to_gd - gemini-3-flash-preview


  0%|          | 0/908 [00:00<?, ?it/s]

Json decode error:
{"translation_text": "Faclair Armstrong (2). Tha mi ag obair tron aibidil ann an seann fhaclair Armstrong. Tha mi a\u2019 coimhead airson faclan-fillte neo-\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0000\u0

  0%|          | 0/908 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 


 



In [ ]:
task_results_df_dict = {
    task: make_combined_df(
        df_dict.values()
    ) for task, df_dict in batch_results.items()
}

In [ ]:
en_transcripts = list(podcasts_ds.map(
    lambda x: {
        "gold_translation": format_article(x["en_title"], x["en_text"])
    }
)["gold_translation"])

gd_transcripts = list(podcasts_ds.map(
    lambda x: {
        "gold_translation": format_article(x["gd_title"], x["gd_text"])
    }
)["gold_translation"])

page_ids = list(podcasts_ds["page_id"])

task_results_df_dict["gd_to_en"]["gold_translation"] = en_transcripts
task_results_df_dict["gd_to_en"]["page_id"] = page_ids
task_results_df_dict["en_to_gd"]["gold_translation"] = gd_transcripts
task_results_df_dict["en_to_gd"]["page_id"] = page_ids

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

In [ ]:
for task_name, results_df in task_results_df_dict.items():
    results_df.to_csv(f"translation_{task_name}_results.csv", index=False)

# Run open QA

In [ ]:
openqa_batch_ids = {}

for lang in ds_dict["open_questions"].keys():
    openqa_batch_ids[lang] = {}
    for model_name in models_to_evaluate:
        print(f"OpenQA ({lang}) - {model_name}")
        openqa_batch_ids[lang][model_name] = send_open_mcq_batch(
            ds_dict["open_questions"][lang], model_name
        )

openqa_batch_ids

OpenQA (en) - gemini-3-pro-preview
OpenQA (en) - gemini-3-flash-preview
OpenQA (en) - gemini-2.5-flash
OpenQA (en) - claude-haiku-4-5-20251001
OpenQA (en) - claude-opus-4-6
OpenQA (en) - deepseek-ai/DeepSeek-R1


Uploading file together_1770819610.0931897.jsonl: 100%|██████████| 1.18M/1.18M [00:00<00:00, 2.84MB/s]


OpenQA (en) - openai/gpt-oss-120b


Uploading file together_1770819612.7911193.jsonl: 100%|██████████| 1.18M/1.18M [00:00<00:00, 2.60MB/s]


OpenQA (en) - OpenAI/gpt-oss-20B


...

Uploading file together_1770819660.261795.jsonl: 100%|██████████| 1.26M/1.26M [00:00<00:00, 3.03MB/s]


OpenQA (gd) - Qwen/Qwen3-Next-80B-A3B-Thinking


Uploading file together_1770819662.780188.jsonl: 100%|██████████| 1.24M/1.24M [00:00<00:00, 2.93MB/s]


OpenQA (gd) - gpt-4o-2024-08-06
OpenQA (gd) - gpt-4o-mini-2024-07-18
OpenQA (gd) - gpt-4.1-2025-04-14
OpenQA (gd) - gpt-4.1-mini-2025-04-14
OpenQA (gd) - gpt-4.1-nano-2025-04-14
OpenQA (gd) - gpt-5.2-2025-12-11
OpenQA (gd) - gpt-5-2025-08-07
OpenQA (gd) - gpt-5-mini-2025-08-07
OpenQA (gd) - gpt-5-nano-2025-08-07


{'en': {'gemini-3-pro-preview': 'batches/hpb8lql3cz4olvdf2wby4wxba3gv6zsyhhno',
  'gemini-3-flash-preview': 'batches/za602tl1g1l71gsbu29dmmvj8c9275e1pfge',
  'gemini-2.5-flash': 'batches/imr17u7qdwavc0gz85vrvoppxsaccftjccph',
  'claude-haiku-4-5-20251001': 'msgbatch_01FzsykgJz3YLB8QUhGgXtR9',
  'claude-opus-4-6': 'msgbatch_014KR2fSGLjdHVafbVjRHE8F',
  'deepseek-ai/DeepSeek-R1': 'a5757b5a-f3a9-44cb-a19d-42788bc050e7',
  'openai/gpt-oss-120b': 'c1816cce-7835-4265-8116-82c45b716a1d',
  'OpenAI/gpt-oss-20B': 'a0df0a00-ddf9-49d3-a938-83b19c84eeb9',
  'moonshotai/Kimi-K2.5': '1a4dbfb0-6af7-4038-94d5-1c4ed53b4974',
  'zai-org/GLM-4.7': 'df4e0dc4-c9ab-455c-bbb0-ec552cd91086',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '2ab2edc8-e99d-49cb-abf9-441bd73c9f27',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': 'e64e382d-6d10-4785-84d8-108312a63a2d',
  'gpt-4o-2024-08-06': 'batch_698c90300b708190bca7053b9d5b20a2',
  'gpt-4o-mini-2024-07-18': 'batch_698c9030e8cc81908583c66d45622e82',
  'gpt-4.1-2025

In [ ]:
openqa_batch_ids = {'en': {'gemini-3-pro-preview': 'batches/hpb8lql3cz4olvdf2wby4wxba3gv6zsyhhno',
  'gemini-3-flash-preview': 'batches/za602tl1g1l71gsbu29dmmvj8c9275e1pfge',
  'gemini-2.5-flash': 'batches/imr17u7qdwavc0gz85vrvoppxsaccftjccph',
  'claude-haiku-4-5-20251001': 'msgbatch_01FzsykgJz3YLB8QUhGgXtR9',
  'claude-opus-4-6': 'msgbatch_014KR2fSGLjdHVafbVjRHE8F',
  'deepseek-ai/DeepSeek-R1': 'a5757b5a-f3a9-44cb-a19d-42788bc050e7',
  'openai/gpt-oss-120b': 'c1816cce-7835-4265-8116-82c45b716a1d',
  'OpenAI/gpt-oss-20B': 'a0df0a00-ddf9-49d3-a938-83b19c84eeb9',
  'zai-org/GLM-4.7': 'df4e0dc4-c9ab-455c-bbb0-ec552cd91086',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '2ab2edc8-e99d-49cb-abf9-441bd73c9f27',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': 'e64e382d-6d10-4785-84d8-108312a63a2d',
  'gpt-4o-2024-08-06': 'batch_698c90300b708190bca7053b9d5b20a2',
  'gpt-4o-mini-2024-07-18': 'batch_698c9030e8cc81908583c66d45622e82',
  'gpt-4.1-2025-04-14': 'batch_698c9031d7f081909b153bb1cc9cc3dc',
  'gpt-4.1-mini-2025-04-14': 'batch_698c903303308190a702016c6dedfa95',
  'gpt-4.1-nano-2025-04-14': 'batch_698c9033ff5081909d2c559a93b4cc3f',
  'gpt-5.2-2025-12-11': 'batch_698c9034c2c481909e958eb825864a64',
  'gpt-5-2025-08-07': 'batch_698c9035c65881908aea50ab177d09bb',
  'gpt-5-mini-2025-08-07': 'batch_698c90370ef48190a87599e5f9df9b09',
  'gpt-5-nano-2025-08-07': 'batch_698c903882ac8190a79ca2028063d59c'},
 'gd': {'gemini-3-pro-preview': 'batches/bdsfqcdik2ft623eeksy2gqivptlvog4bf8c',
  'gemini-3-flash-preview': 'batches/miqd58i36ms6paic52qa4vo86aeo55qpk4zp',
  'gemini-2.5-flash': 'batches/ovna36ko18ariuu4wm4g52lmznbbxqzk9dnw',
  'claude-haiku-4-5-20251001': 'msgbatch_012TrpThEWPPQZLd9g75ri51',
  'claude-opus-4-6': 'msgbatch_015bNdiYfWUcCEYjGPQ6J1p1',
  'deepseek-ai/DeepSeek-R1': 'd8c90864-2e2f-4f11-9a01-d76b1177e56e',
  'openai/gpt-oss-120b': 'c6364c86-11c4-420f-99ee-e77d05b09f3f',
  'OpenAI/gpt-oss-20B': 'a2b8fc3b-a585-46ad-b2ff-a86da19180c5',
  'zai-org/GLM-4.7': 'a31137b0-5970-4906-9a80-8642ea3d805b',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '6ba5c0b5-6211-4322-80de-88251c2d8149',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': 'bdb4e06c-f45d-4aa7-8a64-dc60b7ee9a9e',
  'gpt-4o-2024-08-06': 'batch_698c905275e48190a5820aea06211fcd',
  'gpt-4o-mini-2024-07-18': 'batch_698c9053f8848190a2de676f65327a4e',
  'gpt-4.1-2025-04-14': 'batch_698c9055a790819089ddd3db59b17e21',
  'gpt-4.1-mini-2025-04-14': 'batch_698c9056bf10819097527241388495f2',
  'gpt-4.1-nano-2025-04-14': 'batch_698c9057c20c8190b2fad7b492d094cd',
  'gpt-5.2-2025-12-11': 'batch_698c90588d78819093ec3c7a5aa2be00',
  'gpt-5-2025-08-07': 'batch_698c9059957c819086a7a82059046326',
  'gpt-5-mini-2025-08-07': 'batch_698c905a7b5081908b25694be1321f26',
  'gpt-5-nano-2025-08-07': 'batch_698c905bb19481909153708626ec3610'}}

In [ ]:
for task, batch_ids in openqa_batch_ids.items():
    for model_name, batch_id in batch_ids.items():
        print(f"{task} - {model_name}")
        print(get_batch_status(model_name, batch_id))

en - gemini-3-pro-preview
JOB_STATE_SUCCEEDED
en - gemini-3-flash-preview
JOB_STATE_SUCCEEDED
en - gemini-2.5-flash
JOB_STATE_SUCCEEDED
en - claude-haiku-4-5-20251001
ended
en - claude-opus-4-6
ended
en - deepseek-ai/DeepSeek-R1
COMPLETED
en - openai/gpt-oss-120b
COMPLETED
en - OpenAI/gpt-oss-20B
COMPLETED
en - moonshotai/Kimi-K2.5
FAILED
en - zai-org/GLM-4.7
COMPLETED
en - meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8
COMPLETED
en - Qwen/Qwen3-Next-80B-A3B-Thinking
COMPLETED
en - gpt-4o-2024-08-06
completed
en - gpt-4o-mini-2024-07-18
completed
en - gpt-4.1-2025-04-14
completed
en - gpt-4.1-mini-2025-04-14
completed
en - gpt-4.1-nano-2025-04-14
completed
en - gpt-5.2-2025-12-11
completed
en - gpt-5-2025-08-07
completed
en - gpt-5-mini-2025-08-07
completed
en - gpt-5-nano-2025-08-07
completed
gd - gemini-3-pro-preview
JOB_STATE_SUCCEEDED
gd - gemini-3-flash-preview
JOB_STATE_SUCCEEDED
gd - gemini-2.5-flash
JOB_STATE_SUCCEEDED
gd - claude-haiku-4-5-20251001
ended
gd - claude-opus-4-

In [ ]:
import pandas as pd

batch_results = {}

for task, batch_ids in openqa_batch_ids.items():
    batch_results[task] = {}
    for model_name, batch_id in batch_ids.items():
        print(f"OpenQA {task} - {model_name}")
        batch_result = retrieve_batch(batch_id, model_name)
        batch_result = [x if isinstance(x, dict) else dict() for x in batch_result]
        model_result_df = pd.DataFrame(
            batch_result
        )
        print(model_result_df.shape)
        model_result_df.columns = [f"{c} - {model_name}" for c in model_result_df.columns]
        batch_results[task][model_name] = model_result_df

OpenQA en - gemini-3-pro-preview


  0%|          | 0/1087 [00:00<?, ?it/s]

(1087, 1)
OpenQA en - gemini-3-flash-preview


  0%|          | 0/1087 [00:00<?, ?it/s]

(1087, 1)
OpenQA en - gemini-2.5-flash


...

(1087, 1)
OpenQA gd - gemini-3-flash-preview


  0%|          | 0/1087 [00:00<?, ?it/s]

(1087, 1)
OpenQA gd - gemini-2.5-flash


  0%|          | 0/1087 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 
 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

  
 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 

 
 

 
 

 
 


 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 
 
 

 
 

 
 

 

 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 

 
 


In [ ]:
openqa_results_df_dict = {
    task: make_combined_df(
        df_dict.values()
    ) for task, df_dict in batch_results.items()
}

In [ ]:
for lang, ds in ds_dict["open_questions"].items():
    results_df = openqa_results_df_dict[lang]
    joined_df = results_df.join(ds.to_pandas())
    joined_df.to_csv(f"openqa_{lang}.csv")

# Run closed QA with text input

In [ ]:
closedqa_batch_ids = {}

for lang in ds_dict["closed_questions"].keys():
    closedqa_batch_ids[lang] = {}
    for model_name in models_to_evaluate:
        print(f"ClosedQA ({lang}) - {model_name}")
        closedqa_batch_ids[lang][model_name] = send_closed_mcq_batch(
            ds_dict["closed_questions"][lang], model_name
        )
        closedqa_batch_ids[lang][model_name + "_baseline"] = send_open_mcq_batch(
            ds_dict["closed_questions"][lang], model_name
        )

closedqa_batch_ids

ClosedQA (en) - gemini-3-pro-preview
ClosedQA (en) - gemini-3-flash-preview
ClosedQA (en) - gemini-2.5-flash
ClosedQA (en) - claude-haiku-4-5-20251001
ClosedQA (en) - claude-opus-4-6
ClosedQA (en) - deepseek-ai/DeepSeek-R1


Uploading file together_1770819888.223355.jsonl: 100%|██████████| 3.23M/3.23M [00:00<00:00, 7.52MB/s]
Uploading file together_1770819892.5173461.jsonl: 100%|██████████| 1.08M/1.08M [00:00<00:00, 3.12MB/s]


ClosedQA (en) - openai/gpt-oss-120b


Uploading file together_1770819895.4493976.jsonl: 100%|██████████| 3.23M/3.23M [00:00<00:00, 7.55MB/s]
Uploading file together_1770819897.9564157.jsonl: 100%|██████████| 1.08M/1.08M [00:00<00:00, 2.89MB/s]


ClosedQA (en) - OpenAI/gpt-oss-20B


...

Uploading file together_1770819999.1760006.jsonl: 100%|██████████| 3.33M/3.33M [00:00<00:00, 8.43MB/s]
Uploading file together_1770820001.327923.jsonl: 100%|██████████| 1.18M/1.18M [00:00<00:00, 3.40MB/s]


ClosedQA (gd) - Qwen/Qwen3-Next-80B-A3B-Thinking


Uploading file together_1770820003.391264.jsonl: 100%|██████████| 3.31M/3.31M [00:00<00:00, 8.87MB/s]
Uploading file together_1770820006.2592914.jsonl: 100%|██████████| 1.16M/1.16M [00:00<00:00, 3.19MB/s]


ClosedQA (gd) - gpt-4o-2024-08-06
ClosedQA (gd) - gpt-4o-mini-2024-07-18
ClosedQA (gd) - gpt-4.1-2025-04-14
ClosedQA (gd) - gpt-4.1-mini-2025-04-14
ClosedQA (gd) - gpt-4.1-nano-2025-04-14
ClosedQA (gd) - gpt-5.2-2025-12-11
ClosedQA (gd) - gpt-5-2025-08-07
ClosedQA (gd) - gpt-5-mini-2025-08-07
ClosedQA (gd) - gpt-5-nano-2025-08-07


{'en': {'gemini-3-pro-preview': 'batches/7s4bt935s0p8q91nwcuba5f1r8mb3jwaa6f8',
  'gemini-3-pro-preview_baseline': 'batches/wlg1baa1kwxkou092vf376nk16hya2uprosd',
  'gemini-3-flash-preview': 'batches/thhkypijs06n6d7ld39vt0tukaj0zj0fog1e',
  'gemini-3-flash-preview_baseline': 'batches/iz0fx0vjtfdziehfwmcvmkjeapuelqdnmo3e',
  'gemini-2.5-flash': 'batches/gedmswe0uj3i9mkjb9xb925kvkuxrnyhnjsk',
  'gemini-2.5-flash_baseline': 'batches/ot3272w8jka0gfbgugipx2uddhk9zrff9f8l',
  'claude-haiku-4-5-20251001': 'msgbatch_0151bdbWeBtg17DgJawtksBM',
  'claude-haiku-4-5-20251001_baseline': 'msgbatch_01JwFFC7yEgkmS4X4VbNWC3D',
  'claude-opus-4-6': 'msgbatch_01CX8ipSnjK1HdDsgmyvu1Bu',
  'claude-opus-4-6_baseline': 'msgbatch_01HDmf3oRVvMDGLDPjMmMxGp',
  'deepseek-ai/DeepSeek-R1': 'ce966081-a073-4aff-b7ef-7f9292b1895f',
  'deepseek-ai/DeepSeek-R1_baseline': 'cc48089e-06cb-4736-9bf5-c75edd76460d',
  'openai/gpt-oss-120b': '6b193e54-a691-4a0a-b12e-a499fe790b56',
  'openai/gpt-oss-120b_baseline': 'bcaebf90-7

In [ ]:
closedqa_batch_ids = {'en': {'gemini-3-pro-preview': 'batches/7s4bt935s0p8q91nwcuba5f1r8mb3jwaa6f8',
  'gemini-3-pro-preview_baseline': 'batches/wlg1baa1kwxkou092vf376nk16hya2uprosd',
  'gemini-3-flash-preview': 'batches/thhkypijs06n6d7ld39vt0tukaj0zj0fog1e',
  'gemini-3-flash-preview_baseline': 'batches/iz0fx0vjtfdziehfwmcvmkjeapuelqdnmo3e',
  'gemini-2.5-flash': 'batches/gedmswe0uj3i9mkjb9xb925kvkuxrnyhnjsk',
  'gemini-2.5-flash_baseline': 'batches/ot3272w8jka0gfbgugipx2uddhk9zrff9f8l',
  'claude-haiku-4-5-20251001': 'msgbatch_0151bdbWeBtg17DgJawtksBM',
  'claude-haiku-4-5-20251001_baseline': 'msgbatch_01JwFFC7yEgkmS4X4VbNWC3D',
  'claude-opus-4-6': 'msgbatch_01CX8ipSnjK1HdDsgmyvu1Bu',
  'claude-opus-4-6_baseline': 'msgbatch_01HDmf3oRVvMDGLDPjMmMxGp',
  'deepseek-ai/DeepSeek-R1': 'ce966081-a073-4aff-b7ef-7f9292b1895f',
  'deepseek-ai/DeepSeek-R1_baseline': 'cc48089e-06cb-4736-9bf5-c75edd76460d',
  'openai/gpt-oss-120b': '6b193e54-a691-4a0a-b12e-a499fe790b56',
  'openai/gpt-oss-120b_baseline': 'bcaebf90-7540-48ff-aa46-2f3d763616cd',
  'OpenAI/gpt-oss-20B': '7f002a1c-e22f-439a-8ec6-3f539885cf56',
  'OpenAI/gpt-oss-20B_baseline': 'e91fdbd0-d963-43be-a34f-72d777edbb00',
  # 'moonshotai/Kimi-K2.5': '21006207-fcaf-4523-bc2b-9f61f9876802',
  # 'moonshotai/Kimi-K2.5_baseline': '670bfe07-f87e-4f7b-ba2e-e240f7c7b71c',
  'zai-org/GLM-4.7': '8ea2fdd2-86b1-4f82-b56a-991479c44219',
  'zai-org/GLM-4.7_baseline': '36f8d82b-3041-41d2-b287-b0bfba4b0b8f',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '6053f0e0-d753-463f-8053-6b44e5a80402',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8_baseline': '9c8507e1-fc46-4c37-b327-5b2ab1d11ae1',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': 'eceeb3e3-afc3-472a-92b7-236251b54327',
  'Qwen/Qwen3-Next-80B-A3B-Thinking_baseline': 'ff12ac35-55f3-4e96-bb50-baccc957de98',
  'gpt-4o-2024-08-06': 'batch_698c915d6cf48190b633157b573eb289',
  'gpt-4o-2024-08-06_baseline': 'batch_698c915e914881908bd671af30337209',
  'gpt-4o-mini-2024-07-18': 'batch_698c9160bd308190a27ea347e65b67df',
  'gpt-4o-mini-2024-07-18_baseline': 'batch_698c91618ae88190aefc266a34a48a53',
  'gpt-4.1-2025-04-14': 'batch_698c9162dcdc8190be6180cca21f7fae',
  'gpt-4.1-2025-04-14_baseline': 'batch_698c9163ddd48190ace7cfa32046731c',
  'gpt-4.1-mini-2025-04-14': 'batch_698c9165161081909420c97c66db207d',
  'gpt-4.1-mini-2025-04-14_baseline': 'batch_698c916628788190ab15715edde36b86',
  'gpt-4.1-nano-2025-04-14': 'batch_698c916787c08190bc6654f33619457f',
  'gpt-4.1-nano-2025-04-14_baseline': 'batch_698c9168fc688190abbf97aa1fdcb2bc',
  'gpt-5.2-2025-12-11': 'batch_698c916a75c08190bfb7ce59ea026428',
  'gpt-5.2-2025-12-11_baseline': 'batch_698c916bdc048190a6e9924a1f2f8267',
  'gpt-5-2025-08-07': 'batch_698c916d0fd08190aa15c7d008f04ef4',
  'gpt-5-2025-08-07_baseline': 'batch_698c916df9d08190b4abe98677370673',
  'gpt-5-mini-2025-08-07': 'batch_698c916faa648190afd3c3a221aec256',
  'gpt-5-mini-2025-08-07_baseline': 'batch_698c9170c8c88190af62d556b4031dd5',
  'gpt-5-nano-2025-08-07': 'batch_698c9172bcb48190854f7dee8bf036cd',
  'gpt-5-nano-2025-08-07_baseline': 'batch_698c9173a47c819083a0948edc683757'},
 'gd': {'gemini-3-pro-preview': 'batches/uzo1o9zwyo10xo15zdcsoys2h857ww9baej3',
  'gemini-3-pro-preview_baseline': 'batches/vubgpmvempglgkmqkqfa052k3u6j26d5s6up',
  'gemini-3-flash-preview': 'batches/fr94z8xrvryqstb1kna9eopjjdzh1jri0et6',
  'gemini-3-flash-preview_baseline': 'batches/w6k4awho2tcq1ade6ke8uaa3iobaobmmqh6f',
  'gemini-2.5-flash': 'batches/lpy5lkeuwhn4ysr6hgf7e1jvdkyob6yzitnw',
  'gemini-2.5-flash_baseline': 'batches/22hs84d9d5fxxg3oe7tmube4jzmjreywk41n',
  'claude-haiku-4-5-20251001': 'msgbatch_01JNe6xnF9gurP9ho81JsmcZ',
  'claude-haiku-4-5-20251001_baseline': 'msgbatch_01Ak4qhDdHeG5cdV7jZ9DHLg',
  'claude-opus-4-6': 'msgbatch_0134onMiT25cTG8t9zqV5ESv',
  'claude-opus-4-6_baseline': 'msgbatch_01DVvtn9HnxeUtAwgZnM2dJG',
  'deepseek-ai/DeepSeek-R1': '5487b244-e9d2-4f10-8808-a9d97e3959fc',
  'deepseek-ai/DeepSeek-R1_baseline': '9e741055-fd3c-432d-9a43-c2c1b1ab0f4e',
  'openai/gpt-oss-120b': '5342433b-9d3f-4a04-84a5-71fb50fa132a',
  'openai/gpt-oss-120b_baseline': '220cc3ea-333a-4664-84c3-15c1cbd7a4f9',
  'OpenAI/gpt-oss-20B': '1728d8a8-c4ea-4e78-9fda-af89d2972c71',
  'OpenAI/gpt-oss-20B_baseline': '8d076149-efe6-4602-a27e-ee3909ca60ad',
  # 'moonshotai/Kimi-K2.5': 'f604e1fd-f6fc-458b-9b63-bbfdc46cd5b7',
  # 'moonshotai/Kimi-K2.5_baseline': '77214d5e-5f54-471f-84ba-f9373d5149e0',
  'zai-org/GLM-4.7': 'df5081c9-a45f-4f7f-859f-95aadd4d51db',
  'zai-org/GLM-4.7_baseline': 'b3b99c9e-ece5-4104-bf1a-3fa10289b2e2',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': '105bfe80-cdfe-4f85-b4eb-b2d9d011adf5',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8_baseline': 'ae86bdd2-6d98-42f1-94f9-7ce0818a507a',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': '4df74627-d86d-412e-a454-b51742065c28',
  'Qwen/Qwen3-Next-80B-A3B-Thinking_baseline': '22e66b32-5067-4ce9-99a7-7fac38f1b3b5',
  'gpt-4o-2024-08-06': 'batch_698c91a9c45c8190bb4bac97981e1e84',
  'gpt-4o-2024-08-06_baseline': 'batch_698c91aac9948190a2b57a3a80c56e08',
  'gpt-4o-mini-2024-07-18': 'batch_698c91ac42b48190a68d579f682e5d23',
  'gpt-4o-mini-2024-07-18_baseline': 'batch_698c91adca688190906eb809fd8f7eeb',
  'gpt-4.1-2025-04-14': 'batch_698c91af5594819087eb3988e73dc416',
  'gpt-4.1-2025-04-14_baseline': 'batch_698c91b0741881908e0a87ca59c4acb3',
  'gpt-4.1-mini-2025-04-14': 'batch_698c91b21c0c8190a1ec693124e3c3ca',
  'gpt-4.1-mini-2025-04-14_baseline': 'batch_698c91b2f6ac8190afc8a2a4a590bf28',
  'gpt-4.1-nano-2025-04-14': 'batch_698c91b4a9c48190b427ece776dd9fda',
  'gpt-4.1-nano-2025-04-14_baseline': 'batch_698c91b656a0819098acef2217d131aa',
  'gpt-5.2-2025-12-11': 'batch_698c91b7b3448190b9ac49d23caf82f3',
  'gpt-5.2-2025-12-11_baseline': 'batch_698c91b8e37481908e50ab4094fde83a',
  'gpt-5-2025-08-07': 'batch_698c91ba897c819097daef2c02a5b712',
  'gpt-5-2025-08-07_baseline': 'batch_698c91bb652c8190a81302169033564e',
  'gpt-5-mini-2025-08-07': 'batch_698c91bd862c8190af8b71c901b2c6cb',
  'gpt-5-mini-2025-08-07_baseline': 'batch_698c91bef7308190abeb5ddfab51b8ef',
  'gpt-5-nano-2025-08-07': 'batch_698c91c123dc8190b1e2b7396204c859',
  'gpt-5-nano-2025-08-07_baseline': 'batch_698c91c203fc8190a3e17bdb1995b7df'}}

In [ ]:
for task, batch_ids in closedqa_batch_ids.items():
    for model_name, batch_id in batch_ids.items():
        print(f"{task} - {model_name}")
        print(get_batch_status(model_name, batch_id))

en - gemini-3-pro-preview
JOB_STATE_SUCCEEDED
en - gemini-3-pro-preview_baseline
JOB_STATE_SUCCEEDED
en - gemini-3-flash-preview
JOB_STATE_SUCCEEDED
en - gemini-3-flash-preview_baseline
JOB_STATE_SUCCEEDED
en - gemini-2.5-flash
JOB_STATE_SUCCEEDED
en - gemini-2.5-flash_baseline
JOB_STATE_SUCCEEDED
en - claude-haiku-4-5-20251001
ended
en - claude-haiku-4-5-20251001_baseline
ended
en - claude-opus-4-6
ended
en - claude-opus-4-6_baseline
ended
en - deepseek-ai/DeepSeek-R1
COMPLETED
en - deepseek-ai/DeepSeek-R1_baseline
COMPLETED
en - openai/gpt-oss-120b
COMPLETED
en - openai/gpt-oss-120b_baseline
COMPLETED
en - OpenAI/gpt-oss-20B
COMPLETED
en - OpenAI/gpt-oss-20B_baseline
COMPLETED
en - moonshotai/Kimi-K2.5
FAILED
en - moonshotai/Kimi-K2.5_baseline
FAILED
en - zai-org/GLM-4.7
COMPLETED
en - zai-org/GLM-4.7_baseline
COMPLETED
en - meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8
COMPLETED
en - meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8_baseline
COMPLETED
en - Qwen/Qwen3-Next-80B-A3

In [ ]:
import pandas as pd

batch_results = {}

for task, batch_ids in closedqa_batch_ids.items():
    batch_results[task] = {}
    for model_name, batch_id in batch_ids.items():
        print(f"ClosedQA {task} - {model_name}")
        batch_result = retrieve_batch(batch_id, model_name)
        batch_result = [x if isinstance(x, dict) else dict() for x in batch_result]
        model_result_df = pd.DataFrame(
            batch_result
        )
        print(model_result_df.shape)
        model_result_df.columns = [f"{c} - {model_name}" for c in model_result_df.columns]
        batch_results[task][model_name] = model_result_df

ClosedQA en - gemini-3-pro-preview


  0%|          | 0/908 [00:00<?, ?it/s]

(908, 1)
ClosedQA en - gemini-3-pro-preview_baseline


  0%|          | 0/908 [00:00<?, ?it/s]

(908, 1)
ClosedQA en - gemini-3-flash-preview


...

(908, 1)
ClosedQA gd - gemini-2.5-flash


  0%|          | 0/908 [00:00<?, ?it/s]

(908, 1)
ClosedQA gd - gemini-2.5-flash_baseline


  0%|          | 0/908 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						
   						

In [ ]:
closedqa_results_df_dict = {
    task: make_combined_df(
        df_dict.values()
    ) for task, df_dict in batch_results.items()
}

In [ ]:
for lang, ds in ds_dict["closed_questions"].items():
    results_df = closedqa_results_df_dict[lang]
    joined_df = results_df.join(ds.to_pandas())
    joined_df.to_csv(f"closedqa_{lang}.csv")

# Manually created questions

In [ ]:
system_messages = {
    "gd": """Is e cuidiche fiosrachail a tha annad agus is urrainn dhut a h-uile seòrsa ceist a fhreagairt.
Tagh an romhainn cheart.
Na cuir a-mach ach litir na freagairte ceirte, as aonais mìneachadh no pongachadh sam bith eile.

Eisimpleirean:

Dè 'n dath a th' air an iarmailt? ['A. gorm', 'B. buidhe', 'C. uaine']. Na cuir a-mach ach 'A'

Cò 'n dùthaich às na leanas a tha ann an Afraca? ['A. A' Ghearmailt', 'B. Meagsago', 'C. Nigèiria']. Na cuir a-mach ach 'C'""",

    "en": """You are a knowledgeable assistant that can answer all kinds of questions.
Please select the correct option.
Output ONLY the letter of the correct option, without any additional explanation or punctuation.

Examples:

What colour is the sky? ['A. blue', 'B. yellow', 'C. green']. Return ONLY 'A'

Which of these countries is in Africa? ['A. Germany', 'B. Mexico', 'C. Nigeria']. Return ONLY 'C'"""

}

manualqa_batch_ids = {}

for lang, sys_msg in system_messages.items():
    manualqa_batch_ids[lang] = {}
    for model_name in models_to_evaluate:
        print(f"Manual QA - {lang} - {model_name}")
        manualqa_batch_ids[lang][model_name] = send_open_mcq_batch(
            manual_ds, model_name, sys_msg=sys_msg
        )

manualqa_batch_ids

Manual QA - en - meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8


Uploading file together_1771606251.6782503.jsonl: 100%|██████████| 150k/150k [00:01<00:00, 116kB/s]


Manual QA - en - Qwen/Qwen3-Next-80B-A3B-Thinking


Uploading file together_1771606256.2393758.jsonl: 100%|██████████| 148k/148k [00:02<00:00, 71.2kB/s]


Manual QA - en - gpt-4o-2024-08-06
Manual QA - en - gpt-4o-mini-2024-07-18
Manual QA - en - gpt-4.1-2025-04-14
Manual QA - en - gpt-4.1-mini-2025-04-14
Manual QA - en - gpt-4.1-nano-2025-04-14
Manual QA - en - gpt-5.2-2025-12-11
Manual QA - en - gpt-5-2025-08-07
Manual QA - en - gpt-5-mini-2025-08-07
Manual QA - en - gpt-5-nano-2025-08-07


{'gd': {'gemini-3-pro-preview': 'batches/34dv29lf15ku2s8h5vnzds5s3xlp6owrjy6z',
  'gemini-3-flash-preview': 'batches/5p3rbnbjl195umj01sm60a5u4prksmn2mstb',
  'gemini-2.5-flash': 'batches/e4iunhoaiwrbi7gom1b9otx9xyxu3pmw8u2h',
  'claude-haiku-4-5-20251001': 'msgbatch_01HEK13YQojABuZyUxGnUpoe',
  'claude-opus-4-6': 'msgbatch_01XCvQ8vHb2k5hHdrDYM35aM',
  'deepseek-ai/DeepSeek-R1': 'af935272-a3b2-48b0-8fda-daafddb334f1',
  'openai/gpt-oss-120b': '66a93c7b-278b-440f-b6aa-38a318b06214',
  'OpenAI/gpt-oss-20B': '3033c40d-1c74-4975-9d9a-1ee8602b92b9',
  'moonshotai/Kimi-K2.5': 'd3acc5f0-26a7-402c-a051-c9ea83e13bf5',
  'zai-org/GLM-4.7': '64daa4df-76b0-4752-8352-2e7f15226741',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'df1bf78c-4e40-456a-a8cd-feb1d998fef5',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': '898abf16-4d21-4415-b58f-e925b251ad72',
  'gpt-4o-2024-08-06': 'batch_69988b7ac1fc8190b7532b8e6a796f86',
  'gpt-4o-mini-2024-07-18': 'batch_69988b7ba210819092dae304cee73ac2',
  'gpt-4.1-2025

In [ ]:
manualqa_batch_ids = {'gd': {'gemini-3-pro-preview': 'batches/34dv29lf15ku2s8h5vnzds5s3xlp6owrjy6z',
  'gemini-3-flash-preview': 'batches/5p3rbnbjl195umj01sm60a5u4prksmn2mstb',
  'gemini-2.5-flash': 'batches/e4iunhoaiwrbi7gom1b9otx9xyxu3pmw8u2h',
  'claude-haiku-4-5-20251001': 'msgbatch_01HEK13YQojABuZyUxGnUpoe',
  'claude-opus-4-6': 'msgbatch_01XCvQ8vHb2k5hHdrDYM35aM',
  'deepseek-ai/DeepSeek-R1': 'af935272-a3b2-48b0-8fda-daafddb334f1',
  'openai/gpt-oss-120b': '66a93c7b-278b-440f-b6aa-38a318b06214',
  'OpenAI/gpt-oss-20B': '3033c40d-1c74-4975-9d9a-1ee8602b92b9',
  'moonshotai/Kimi-K2.5': 'd3acc5f0-26a7-402c-a051-c9ea83e13bf5',
  'zai-org/GLM-4.7': '64daa4df-76b0-4752-8352-2e7f15226741',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'df1bf78c-4e40-456a-a8cd-feb1d998fef5',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': '898abf16-4d21-4415-b58f-e925b251ad72',
  'gpt-4o-2024-08-06': 'batch_69988b7ac1fc8190b7532b8e6a796f86',
  'gpt-4o-mini-2024-07-18': 'batch_69988b7ba210819092dae304cee73ac2',
  'gpt-4.1-2025-04-14': 'batch_69988b7d3ae08190aee88bd8ec0924e7',
  'gpt-4.1-mini-2025-04-14': 'batch_69988b7e502081909424dcfa0b12cda2',
  'gpt-4.1-nano-2025-04-14': 'batch_69988b7f8dc88190b7ca43b4726ddef9',
  'gpt-5.2-2025-12-11': 'batch_69988b806aac8190b610cf76494810a9',
  'gpt-5-2025-08-07': 'batch_69988b8137048190991cc4a50b5aeef0',
  'gpt-5-mini-2025-08-07': 'batch_69988b825a048190824b0a0468f37615',
  'gpt-5-nano-2025-08-07': 'batch_69988b832cc08190b3359d44efd168c3'},
 'en': {'gemini-3-pro-preview': 'batches/1jhpra92i60j13nd24r5e24p1t9t26zcn920',
  'gemini-3-flash-preview': 'batches/c31tw28bpr55pjopvn9o49nm7wgmlregebs4',
  'gemini-2.5-flash': 'batches/oxwpvtvd8kgce5b8pio0mqh5j2ccyl411m3u',
  'claude-haiku-4-5-20251001': 'msgbatch_01BG9zEdsr4KTyGLi5fLFRVp',
  'claude-opus-4-6': 'msgbatch_01GCqTKYdh9hEgb8uBLygeLG',
  'deepseek-ai/DeepSeek-R1': '95110dc3-a4a7-42d8-9dd9-48331c3cfb49',
  'openai/gpt-oss-120b': '53994944-388d-4100-8950-4f0989355007',
  'OpenAI/gpt-oss-20B': '66918c56-0194-4cb1-94e0-58d8eb5b7a35',
  'moonshotai/Kimi-K2.5': 'ffeb8478-6376-47e8-85f4-55f067afb487',
  'zai-org/GLM-4.7': '0c224110-edc0-4727-b3b1-bdc631867548',
  'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': 'd03194e6-e2c7-40c8-9deb-1cce82584816',
  'Qwen/Qwen3-Next-80B-A3B-Thinking': '1828d773-97f8-4221-a55f-352f0fd0c43b',
  'gpt-4o-2024-08-06': 'batch_699890f5dc6c8190b162a2f799719f31',
  'gpt-4o-mini-2024-07-18': 'batch_699890f6f5b48190b1e8b861b1183a3d',
  'gpt-4.1-2025-04-14': 'batch_699890f7d244819087fb48047308e0a3',
  'gpt-4.1-mini-2025-04-14': 'batch_699890f8e6ac8190acced5914745fcc9',
  'gpt-4.1-nano-2025-04-14': 'batch_699890fa84e48190bdafcac1ce13bd75',
  'gpt-5.2-2025-12-11': 'batch_699890fb77d0819094e7dcae58a08b9e',
  'gpt-5-2025-08-07': 'batch_699890fca38081908db40acec50ff560',
  'gpt-5-mini-2025-08-07': 'batch_699890fd844081908613edd8a0815983',
  'gpt-5-nano-2025-08-07': 'batch_699890fe6c148190af897cf958742f61'}}

In [ ]:
for lang, lang_batches in manualqa_batch_ids.items():
    for model_name, batch_id in lang_batches.items():
        print(f"Manual QA - {lang} - {model_name}")
        print(get_batch_status(model_name, batch_id))

Manual QA - gd - gemini-3-pro-preview
JOB_STATE_SUCCEEDED
Manual QA - gd - gemini-3-flash-preview
JOB_STATE_SUCCEEDED
Manual QA - gd - gemini-2.5-flash
JOB_STATE_SUCCEEDED
Manual QA - gd - claude-haiku-4-5-20251001
ended
Manual QA - gd - claude-opus-4-6
ended
Manual QA - gd - deepseek-ai/DeepSeek-R1
COMPLETED
Manual QA - gd - openai/gpt-oss-120b
COMPLETED
Manual QA - gd - OpenAI/gpt-oss-20B
COMPLETED
Manual QA - gd - moonshotai/Kimi-K2.5
FAILED
Manual QA - gd - zai-org/GLM-4.7
COMPLETED
Manual QA - gd - meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8
COMPLETED
Manual QA - gd - Qwen/Qwen3-Next-80B-A3B-Thinking
COMPLETED
Manual QA - gd - gpt-4o-2024-08-06
completed
Manual QA - gd - gpt-4o-mini-2024-07-18
completed
Manual QA - gd - gpt-4.1-2025-04-14
completed
Manual QA - gd - gpt-4.1-mini-2025-04-14
completed
Manual QA - gd - gpt-4.1-nano-2025-04-14
completed
Manual QA - gd - gpt-5.2-2025-12-11
completed
Manual QA - gd - gpt-5-2025-08-07
completed
Manual QA - gd - gpt-5-mini-2025-08-07

In [ ]:
import pandas as pd

batch_results = {}

for lang, batch_ids in manualqa_batch_ids.items():
    batch_results[lang] = {}
    for model_name, batch_id in batch_ids.items():
        print(f"Manual QA - {model_name}")
        try:
            batch_result = retrieve_batch(batch_id, model_name)
        except:
            print(f"Skip {model_name}")
            continue
        batch_result = [x if isinstance(x, dict) else dict() for x in batch_result]
        model_result_df = pd.DataFrame(
            batch_result
        )["single_letter_answer"]
        print(model_result_df.shape)
        # model_result_df.columns = [f"{c} - {model_name}" for c in model_result_df.columns]
        batch_results[lang][model_name] = model_result_df

Manual QA - gemini-3-pro-preview


  0%|          | 0/123 [00:00<?, ?it/s]

(123,)
Manual QA - gemini-3-flash-preview


  0%|          | 0/123 [00:00<?, ?it/s]

(123,)
Manual QA - gemini-2.5-flash


...

(123,)
Manual QA - gemini-3-flash-preview


  0%|          | 0/123 [00:00<?, ?it/s]

(123,)
Manual QA - gemini-2.5-flash


  0%|          | 0/123 [00:00<?, ?it/s]

(123,)
Manual QA - claude-haiku-4-5-20251001
(123,)
Manual QA - claude-opus-4-6
(123,)
Manual QA - deepseek-ai/DeepSeek-R1
(123,)
Manual QA - openai/gpt-oss-120b
(123,)
Manual QA - OpenAI/gpt-oss-20B
Json decode error:

Json decode error:

Json decode error:

(123,)
Manual QA - moonshotai/Kimi-K2.5
Skip moonshotai/Kimi-K2.5
Manual QA - zai-org/GLM-4.7
Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json decode error:

Json d

In [ ]:

for lang, lang_results in batch_results.items():
    print(lang)
    manual_results_df = pd.DataFrame(lang_results)
    manual_results_df["correct_ans"] = list(manual_ds["correct_ans"])
    manual_results_df.to_csv(f"manual_{lang}_qa.csv")

gd
en


In [ ]:
manual_ds[122]

{'id': 123,
 'category': 'colours',
 'sub_category': 'colour - en / gd mismatch',
 'question': "Is ann ___ a tha falt a' phàiste ___.",
 'answer_A': 'bàn',
 'answer_B': 'dearg',
 'answer_C': 'geal',
 'answer_D': 'aotrom',
 'answer_E': None,
 'correct_ans': 'A'}